# Integral Hinge (Living Hinge) Design Calculator
**Plastic Design Calculators — Notebook 2**

---

## Part 1: Theory and Governing Equations

### 1.1 Mechanics of Living Hinges

Integral or **living hinges** exploit the extreme elongation capability of semi-crystalline polymers—predominantly Polypropylene (PP) and High-Density Polyethylene (PE-HD)—to survive millions of flexural cycles without failure. The first bend immediately after mould ejection orients molecular chains along the hinge axis via **cold drawing**, massively increasing localised tensile strength.

Critical to infinite fatigue life is the hinge geometry: the web must form a perfect stress-distributing **semicircle** upon full closure.

### 1.2 Optimal Hinge Length

For a hinge designed to close through angle $\theta$ (degrees) with bend radius $R$:

$$\boxed{L = \frac{\theta}{180} \pi R}$$

For the standard 180° closure this simplifies to $L = \pi R$.

### 1.3 Web Thickness Limits

Empirical bounds for hinge web thickness $T$ relative to the adjacent wall thickness $H$:

$$\frac{H}{8} \leq T \leq \frac{H}{5}$$

Absolute bounds: $0.2 \text{ mm} \leq T \leq 0.5 \text{ mm}$

### 1.4 Fatigue Life — Coffin–Manson / Basquin Combined Model

Because living hinges operate at large local strains, low-cycle fatigue governs. The combined model accounts for both elastic and plastic strain contributions:

$$\boxed{\epsilon_a = \frac{\sigma'_f}{E}(2N_f)^b + \epsilon'_f(2N_f)^c}$$

where:
- $\epsilon_a$ — total strain amplitude [-]
- $\sigma'_f$ — fatigue strength coefficient [Pa]
- $E$ — elastic modulus [Pa]
- $b$ — Basquin exponent (typically $-0.05$ to $-0.12$)
- $\epsilon'_f$ — fatigue ductility coefficient [-]
- $c$ — Coffin–Manson exponent (typically $-0.5$ to $-0.7$)
- $N_f$ — cycles to failure

### 1.5 Outer-Fibre Bending Strain

Peak outer-fibre strain during closure of a hinge with web thickness $T$ and bend radius $R$:

$$\epsilon_{\mathrm{bend}} = \frac{T/2}{R + T/2}$$

### Assumptions
- Pure bending, plane sections remain plane
- Neutral axis at web midplane
- Constant web cross-section (uniform thickness)
- Temperature 23 °C, no UV degradation
- Fatigue data from Hostalen PPR 1042 (PP homopolymer)

---
## Part 2: Variable Definitions and Unit Handling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import brentq

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import FATIGUE_COEFFICIENTS

# ── Geometry ──────────────────────────────────────────────────────────────────
H         = Q_(2.0,  'mm')     # adjacent wall (rigid body) thickness
R_bend    = Q_(0.4,  'mm')     # bend radius to hinge centre axis
theta_deg = 180.0              # closure angle [degrees]

# ── Design target ─────────────────────────────────────────────────────────────
N_target_min = 1_000_000       # minimum required fatigue cycles
N_target_max = 10_000_000      # ideal target fatigue cycles

# ── Material ──────────────────────────────────────────────────────────────────
MAT_KEY  = 'PP_Hostalen'
fat      = FATIGUE_COEFFICIENTS[MAT_KEY]
sigma_f_prime = fat['sigma_f_prime_Pa']   # [Pa]
E_mat    = fat['E_Pa']                    # [Pa]
b_exp    = fat['b']
eps_f_prime = fat['epsilon_f_prime']
c_exp    = fat['c']

print(f"Material  : {fat['description']}")
print(f"σ'f       : {sigma_f_prime/1e6:.1f} MPa")
print(f"E         : {E_mat/1e6:.0f} MPa")
print(f"b, c      : {b_exp}, {c_exp}")
print(f"ε'f       : {eps_f_prime}")

---
## Part 3: Computation Engine

In [ ]:
# ── 3.1  Symbolic equations ───────────────────────────────────────────────────
theta_s, R_s, L_s, T_s, H_s = sp.symbols('theta R L T H', positive=True)
Nf_s, eps_a_s, sig_f_s, E_s, b_s, eps_f_s, c_s = sp.symbols(
    'N_f epsilon_a sigma_f_prime E b epsilon_f_prime c', real=True)

L_eq   = (theta_s / 180) * sp.pi * R_s
eps_bend_eq = (T_s / 2) / (R_s + T_s / 2)
coffin_basquin = (sig_f_s / E_s) * (2*Nf_s)**b_s + eps_f_s * (2*Nf_s)**c_s

print("Hinge length L =")
sp.pprint(L_eq)
print()
print("Outer-fibre bending strain ε_bend =")
sp.pprint(eps_bend_eq)
print()
print("Coffin–Manson / Basquin εa =")
sp.pprint(coffin_basquin)

In [ ]:
def hinge_length(theta_degrees, bend_radius_m, **kwargs):
    """Developed hinge web length for a given closure angle and bend radius.

    Args:
        theta_degrees (float): Closure angle [degrees].
        bend_radius_m (float): Bend radius to web centre [m].
        **kwargs: Reserved for variable-radius profiles.

    Returns:
        float: Required hinge web length [m].
    """
    return (theta_degrees / 180.0) * np.pi * bend_radius_m


def outer_fibre_strain(web_thickness_m, bend_radius_m, **kwargs):
    """Peak outer-fibre bending strain in the hinge web.

    Args:
        web_thickness_m (float | numpy.ndarray): Hinge web thickness [m].
        bend_radius_m (float): Bend radius to web centre [m].
        **kwargs: Reserved for biaxial strain correction factors.

    Returns:
        float | numpy.ndarray: Outer-fibre strain [-].
    """
    return (web_thickness_m / 2.0) / (bend_radius_m + web_thickness_m / 2.0)


def coffin_manson_strain(N_cycles, sigma_f_prime, E, b, epsilon_f_prime, c, **kwargs):
    """Total strain amplitude from the combined Coffin–Manson / Basquin model.

    Args:
        N_cycles (float | numpy.ndarray): Number of reversals to failure Nf.
        sigma_f_prime (float): Fatigue strength coefficient [Pa].
        E (float): Elastic modulus [Pa].
        b (float): Basquin exponent.
        epsilon_f_prime (float): Fatigue ductility coefficient.
        c (float): Coffin–Manson exponent.
        **kwargs: Reserved for mean-stress corrections (Morrow, SWT).

    Returns:
        float | numpy.ndarray: Total strain amplitude εa [-].
    """
    reversal = 2.0 * np.asarray(N_cycles)
    elastic  = (sigma_f_prime / E) * reversal**b
    plastic  = epsilon_f_prime     * reversal**c
    return elastic + plastic


def solve_fatigue_life(strain_amplitude, sigma_f_prime, E, b, epsilon_f_prime, c,
                       N_bracket=(1e2, 1e9), **kwargs):
    """Solve Coffin–Manson for Nf given a strain amplitude using Brent's method.

    Args:
        strain_amplitude (float): Applied total strain amplitude εa [-].
        sigma_f_prime, E, b, epsilon_f_prime, c: Fatigue model parameters.
        N_bracket (tuple): Search bracket (N_low, N_high).
        **kwargs: Forwarded to coffin_manson_strain.

    Returns:
        float: Cycles to failure Nf.

    Raises:
        ValueError: If solution is outside the provided N_bracket.
    """
    def residual(Nf):
        return coffin_manson_strain(Nf, sigma_f_prime, E, b, epsilon_f_prime, c) - strain_amplitude

    f_lo = residual(N_bracket[0])
    f_hi = residual(N_bracket[1])
    if f_lo * f_hi > 0:
        raise ValueError(
            f"No root in [{N_bracket[0]:.0e}, {N_bracket[1]:.0e}] for εa={strain_amplitude:.4f}. "
            "Strain may exceed or be below material data range."
        )
    return brentq(residual, N_bracket[0], N_bracket[1], xtol=1.0)


print("Functions defined.")

In [ ]:
# ── 3.2  Numerical evaluation ─────────────────────────────────────────────────

H_m    = strip_units(H.to('meter'))
R_m    = strip_units(R_bend.to('meter'))

# Hinge length
L_hinge_m = hinge_length(theta_deg, R_m)

# Thickness bounds (dynamic scaling from wall)
T_min_m = max(H_m / 8.0, 0.2e-3)
T_max_m = min(H_m / 5.0, 0.5e-3)
T_nom_m = (T_min_m + T_max_m) / 2.0   # nominal design value

# Outer-fibre strain at nominal thickness
eps_bend = outer_fibre_strain(T_nom_m, R_m)

# Fatigue life at this strain amplitude
Nf = solve_fatigue_life(eps_bend, sigma_f_prime, E_mat, b_exp, eps_f_prime, c_exp)

print(f"Hinge length L          : {L_hinge_m*1e3:.3f} mm")
print(f"Nominal web thickness T : {T_nom_m*1e3:.3f} mm  (range: {T_min_m*1e3:.3f}–{T_max_m*1e3:.3f} mm)")
print(f"Outer-fibre strain εa   : {eps_bend*100:.3f} %")
print(f"Predicted fatigue life  : {Nf:,.0f} cycles")

# Parametric sweep: Nf vs. web thickness (vectorised)
T_sweep   = np.linspace(0.15e-3, 0.55e-3, 300)
eps_sweep = outer_fibre_strain(T_sweep, R_m)
Nf_sweep  = np.array([solve_fatigue_life(e, sigma_f_prime, E_mat, b_exp, eps_f_prime, c_exp)
                       if e > 0 else np.nan for e in eps_sweep])

---
## Part 4: Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: Fatigue life (Nf) vs web thickness ────────────────────────────────
ax1 = axes[0]
ax1.semilogy(T_sweep * 1e3, Nf_sweep, color='steelblue', lw=2)
ax1.axhline(N_target_min, color='tomato',    ls='--', lw=1.5, label=f'Min target: {N_target_min/1e6:.0f}M cycles')
ax1.axhline(N_target_max, color='limegreen', ls='--', lw=1.5, label=f'Ideal target: {N_target_max/1e6:.0f}M cycles')
ax1.axvline(T_nom_m*1e3, color='darkorange', ls=':', lw=1.5, label=f'Nominal T={T_nom_m*1e3:.3f} mm')
ax1.axvspan(T_min_m*1e3, T_max_m*1e3, alpha=0.12, color='green', label='Recommended range')
ax1.set_xlabel('Web Thickness T [mm]')
ax1.set_ylabel('Cycles to Failure Nf  [—]')
ax1.set_title(f'Living Hinge Fatigue Life vs Thickness\n({fat["description"]})')
ax1.legend(fontsize=8)
ax1.grid(True, which='both', alpha=0.3)

# ── Plot 2: Strain–Life (Coffin–Manson) curve ────────────────────────────────
Nf_arr  = np.logspace(2, 8, 400)
eps_arr = coffin_manson_strain(Nf_arr, sigma_f_prime, E_mat, b_exp, eps_f_prime, c_exp)
eps_el  = (sigma_f_prime / E_mat) * (2*Nf_arr)**b_exp
eps_pl  = eps_f_prime             * (2*Nf_arr)**c_exp

ax2 = axes[1]
ax2.loglog(Nf_arr, eps_arr * 100,  color='steelblue',  lw=2,  label='Total εa')
ax2.loglog(Nf_arr, eps_el  * 100,  color='darkorange', lw=1.5, ls='--', label='Elastic component')
ax2.loglog(Nf_arr, eps_pl  * 100,  color='tomato',     lw=1.5, ls='--', label='Plastic component')
ax2.axvline(Nf,             color='purple',     ls=':', lw=2, label=f'Design point: {Nf:.2e} cycles')
ax2.axhline(eps_bend * 100, color='limegreen',  ls=':', lw=2, label=f'εa = {eps_bend*100:.2f} %')
ax2.set_xlabel('Cycles to Failure Nf  [—]')
ax2.set_ylabel('Strain Amplitude εa  [%]')
ax2.set_title('Strain–Life Curve\n(Coffin–Manson / Basquin)')
ax2.legend(fontsize=8)
ax2.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('02_living_hinge_output.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Design Rule Validation

In [ ]:
pass_fatigue   = Nf >= N_target_min
pass_thickness = T_min_m <= T_nom_m <= T_max_m
pass_absolute  = 0.2e-3 <= T_nom_m <= 0.5e-3

def badge(passed):
    return "\033[92m  PASS  \033[0m" if passed else "\033[91m  FAIL  \033[0m"

print("═" * 62)
print("  DESIGN RULE VALIDATION — LIVING HINGE")
print("═" * 62)
print(f"  Fatigue life:")
print(f"    Nf computed : {Nf:,.0f} cycles")
print(f"    Required    : ≥ {N_target_min:,} cycles")
print(f"    Result      : {badge(pass_fatigue)}")
print()
print(f"  Web thickness — empirical H/5…H/8 rule:")
print(f"    T nominal   : {T_nom_m*1e3:.3f} mm")
print(f"    Range       : {T_min_m*1e3:.3f} – {T_max_m*1e3:.3f} mm")
print(f"    Result      : {badge(pass_thickness)}")
print()
print(f"  Web thickness — absolute mouldability (0.2–0.5 mm):")
print(f"    T nominal   : {T_nom_m*1e3:.3f} mm")
print(f"    Result      : {badge(pass_absolute)}")
print("═" * 62)
overall = pass_fatigue and pass_thickness and pass_absolute
if overall:
    print("  ✓ OVERALL: DESIGN PASSES all living hinge criteria.")
else:
    print("  ✗ OVERALL: DESIGN FAILS — adjust R, T, or material.")
print("═" * 62)